> ⏱ ~20 min

# Scenario

**Section 1.** Before any agent goes near real travelers, you send a mystery shopper. The shopper tries to coax policy violations, unsafe help, or off-topic behavior so you can fix gaps before employees rely on the concierge. This is a **red-team scan**.

Red-team scans can target a Foundry-hosted agent or any callable you build yourself. This notebook targets the **local MAF concierge** — the same multi-agent system you built in notebook 02 — by handing the scanner a Python callable. The scanner's probe prompts still come from the Azure red-team service, and the scan results still upload to your Foundry project for the team to review.


## 2. What you will do

1. Build the same concierge target the eval notebook used.
2. Define a small `target(query, context=None) -> str` callable the red-team scanner can invoke (the SDK passes the probe text as `query=`).
3. Configure `RedTeam` with travel-relevant risk categories and a small objective budget so the scan finishes in workshop time.
4. Run the scan and inspect the summary.

Running the scanner against your own callable means the same red-team workflow works whether you build agents on the hosted Foundry Agent Service or with an open-source framework on top of models deployed in Microsoft Foundry. By owning the target callable, you can also target the **specialist** agents in isolation if you need to debug where a failure originates.

---


In [ ]:
# Suppress experimental/deprecation warnings to keep the learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys, asyncio
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
# Hub-less Microsoft Foundry projects are identified by their project endpoint;
# the SDK uses the right resource path when azure_ai_project is a string URL.
PROJECT_ENDPOINT = os.environ['FOUNDRY_PROJECT_ENDPOINT']
azure_ai_project = PROJECT_ENDPOINT
print('Targeting project:', PROJECT_ENDPOINT)


## 3. Wrap the local concierge as a callable target

The red-team scanner does not care that the concierge is a multi-agent system. It only needs a function that takes one prompt string and returns one string answer. We build the concierge once and reuse it across every probe.


In [ ]:
import threading
from typing import Optional, Dict, Any
from agents import make_chat_client, build_concierge

client = make_chat_client()
concierge = build_concierge(client)

# A single background thread owns one event loop. Both the kernel main
# thread (sanity call) and the red-team scanner's worker threads hand their
# coroutines to this loop via run_coroutine_threadsafe. This keeps MAF's
# telemetry ContextVar tokens set and reset inside the same task context
# and avoids 'asyncio.run() cannot be called from a running event loop'.
_runner_loop = asyncio.new_event_loop()
_runner_thread = threading.Thread(
    target=_runner_loop.run_forever, name='concierge-loop', daemon=True
)
_runner_thread.start()

async def _invoke_concierge(prompt: str):
    return await concierge.run(prompt)

def target(query: str, context: Optional[Dict[str, Any]] = None) -> str:
    """Run the MAF concierge against a single red-team probe.

    The `azure.ai.evaluation.red_team` SDK calls user callables as
    `target(query=<probe>)` (and optionally `context=<dict>` when the
    parameter is present), so the keyword `query` is required. Failures are
    swallowed so a single bad probe does not stop the scan; the error text
    is returned in place of an answer so the scanner can still grade the row.
    """
    try:
        fut = asyncio.run_coroutine_threadsafe(_invoke_concierge(query), _runner_loop)
        response = fut.result()
        return response.text or ''
    except Exception as exc:
        return f'[concierge call failed: {type(exc).__name__}: {exc}]'

print(target(query='Can you help me book a compliant hotel in Seattle?')[:400])


## 4. Configure and launch the red-team scan

`RedTeam` lives in `azure.ai.evaluation.red_team` and is an optional extra. The lab's `requirements.txt` already pulls in `azure-ai-evaluation[redteam]`, so the import should succeed in this environment. If it does not, install the extra and restart the kernel.

Two knobs keep the scan small enough for a workshop room: `risk_categories` (which dimensions to probe) and `num_objectives` (how many probes per category). Increase either when you have time to spare.


In [ ]:
from datetime import datetime
import dataclasses
import json as _json

try:
    from azure.ai.evaluation.red_team import RedTeam, RiskCategory
except ImportError as exc:
    raise RuntimeError(
        'Install azure-ai-evaluation[redteam] (and restart the kernel) before running this cell.'
    ) from exc

red_team = RedTeam(
    azure_ai_project=azure_ai_project,
    credential=credential,
    # Travel-relevant safety dimensions for the enterprise concierge scenario.
    # RiskCategory enum values are required (passing raw strings raises
    # AttributeError inside the scanner). 'indirect_attack' is an attack
    # *strategy*, not a risk category, so it does not belong here.
    risk_categories=[
        RiskCategory.HateUnfairness,
        RiskCategory.Violence,
        RiskCategory.ProhibitedActions,
    ],
    # Eight probes per category keeps the scan under a few minutes for the room.
    num_objectives=8,
)

# Local snapshot of the scan results, alongside the SDK's own scan_output_dir.
RESULTS_DIR = LAB_DIR / 'redteam-results'
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
scan_output_path = RESULTS_DIR / f'maf-concierge-redteam-{timestamp}.json'

# RedTeam.scan is async. We MUST run it on a different event loop than the
# one the concierge uses (`_runner_loop`), or the call deadlocks: the SDK
# invokes our `target(query=...)` synchronously inside an awaited coroutine,
# and that target then blocks on `_runner_loop.run_coroutine_threadsafe(...)
# .result()` — if both ran on the same loop, the loop would be stuck inside
# `target()` waiting for itself to free up. Jupyter cells already run on the
# kernel's event loop, so a top-level `await` here runs the scan on that
# kernel loop while every concierge call still hops to `_runner_loop`.
scan_result = await red_team.scan(
    target=target,
    scan_name='maf-concierge-redteam',
    output_path=str(scan_output_path),
)

# Also persist the in-memory RedTeamResult object as JSON so it is easy to
# diff across runs without parsing the SDK's per-strategy output files.
def _to_jsonable(value):
    if dataclasses.is_dataclass(value):
        return dataclasses.asdict(value)
    if hasattr(value, 'model_dump'):
        return value.model_dump()
    if hasattr(value, '__dict__'):
        return {k: v for k, v in vars(value).items() if not k.startswith('_')}
    return str(value)

summary_path = RESULTS_DIR / f'maf-concierge-redteam-{timestamp}-summary.json'
summary_path.write_text(_json.dumps(scan_result, default=_to_jsonable, indent=2))

print(f'Per-attack outputs : {scan_output_path}')
print(f'Result summary     : {summary_path}')
sdk_dir = getattr(red_team, 'scan_output_dir', None)
if sdk_dir:
    print(f'SDK scan_output_dir: {sdk_dir}')
scan_result


## 5. Review the findings

The scan result contains the probe prompts, the concierge's response to each, and the scanner's judgement (`safe`, `unsafe`, or similar). The full report uploads to the Foundry project; the local return value is a compact summary for the notebook.

Walk through any flagged rows with the room. For each one, decide whether the fix belongs in the concierge instructions, a specialist's instructions, or the booking tools. That is where multi-agent debugging earns its keep — tracing (notebook 03) shows you which agent produced the unsafe answer, and you can rerun that single specialist with a tighter prompt.


## 6. What you confirmed

1. `RedTeam` accepts any Python callable, so the MAF concierge -- built with an open-source framework on top of models deployed in Microsoft Foundry -- can be scanned exactly the same way as a hosted agent.
2. Scans still upload to the Foundry project, keeping safety evidence in the same place as Foundry-hosted agent scans.
3. Combining tracing (notebook 03), evaluations (notebook 04), and red-team (this notebook) gives a complete picture: *how it happened*, *how good*, *how safe*.

You now have a Cohere-powered, multi-agent, locally-runnable, locally-observable, locally-evaluable, and locally-scannable travel concierge.
